In [2]:
import json

def assign_entity_ids(file):

    # load training data from json
    with open(file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # check whether this is an initial annotation
    # a null entity id with Candidate status indicates an initial annotation
    is_candidate_id_null = False
    
    for i in data['annotations']:
        for ent in i[2]['entities']:
            entity_id = ent[0]
    
            if entity_id is None:
                for sts in ent[3]:
                    status = sts[0]
                    
                    if status == 'Candidate':
                        is_candidate_id_null = True
                        
                    break
            
            if is_candidate_id_null:
                break
        
        if is_candidate_id_null:
            break
    
    # assign entity ids for the initial annotation
    if is_candidate_id_null:
        for i in data['annotations']:
            paragraph_id = i[0]
            entity_number = 1
            
            for ent in i[2]['entities']:
                entity_id = f'{paragraph_id}_E{entity_number}'
                ent[0] = entity_id
                entity_number += 1

    if not is_candidate_id_null: 
        # determine the current review number from the ids of the suggested entities (span only) in the previous review
        review_numbers = []
        
        for i in data['annotations']:
            for ent in i[2]['entities']:
                if ent[0] is not None:
                    id_last_part = ent[0].split('_')[-1]
            
                    if id_last_part[0:2] == 'Rv':
                        review_numbers.append(int(id_last_part[2:]))
        
        if bool(review_numbers):
            largest_number = max(review_numbers)
            id_last_part = f'Rv{largest_number+1}'
        else:
            id_last_part = 'Rv1'
    
        # assign entity ids to entities added during the review
        for i in data['annotations']:
            paragraph_id = i[0]
            entity_number = 1
            
            for ent in i[2]['entities']:
                if ent[0] is None:
                    entity_id = f'{paragraph_id}_E{entity_number}_{id_last_part}'
                    ent[0] = entity_id
                    entity_number += 1

    return data


In [6]:
from datetime import datetime

path = '/home/umayer/Work/research/ner_data/annotation/Koshkava_2014'
file = 'Koshkava_2014_8of8_AnNER.json'
annotations = assign_entity_ids(f'{path}/{file}')
timestamp = datetime.now().strftime('%y%m%d%H%M%S')

with open(f'{file.split('.')[0]}_WITH_IDs_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(annotations, f, indent=2, ensure_ascii=False)